# CTGAN Training — Google Colab

Train a CTGAN model on guest profile features from `df_featured.csv`.

**Steps:**
1. Upload `df_featured.csv` from `demand_model/data/`
2. Train CTGAN (GPU-accelerated)
3. Download `ctgan_model.pkl` and `ctgan_metadata.json`
4. Place them in `demand_model/models/`

In [ ]:
# Step 1: Install SDV
!pip install -q sdv

In [ ]:
# Step 2: Upload df_featured.csv
from google.colab import files
uploaded = files.upload()  # Browse and select demand_model/data/df_featured.csv

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import date

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# Column definitions
PROFILE_COLUMNS = [
    'hotel_encoded', 'month_num', 'room_tier', 'los',
    'meal', 'market_segment', 'customer_type',
    'total_guests', 'is_repeated_guest', 'has_special_requests',
    'is_non_refund', 'previous_cancellations', 'lead_time_bucket'
]

DISCRETE_COLUMNS = [
    'hotel_encoded', 'month_num', 'room_tier', 'meal',
    'market_segment', 'customer_type', 'is_repeated_guest',
    'has_special_requests', 'is_non_refund', 'previous_cancellations',
    'lead_time_bucket'
]

CONTINUOUS_COLUMNS = ['los', 'total_guests']

SEGMENT_ENCODE = {
    'Aviation': 0, 'Complementary': 1, 'Corporate': 2, 'Direct': 3,
    'Groups': 4, 'Offline TA/TO': 5, 'Online TA': 6
}
CUSTOMER_TYPE_ENCODE = {
    'Transient': 0, 'Transient-Party': 1, 'Contract': 2, 'Group': 3
}

LOS_CAP = {'city': 9, 'resort': 14}

print('Imports ready')

In [ ]:
# Step 3: Load and prepare data
df = pd.read_csv('df_featured.csv', parse_dates=['arrival_date'])
df_train = df[df['arrival_date'] < '2020-01-01'].copy()
print(f'Training rows: {len(df_train)}')

df_profile = df_train[PROFILE_COLUMNS].copy()
for col in DISCRETE_COLUMNS:
    if col not in ['meal', 'market_segment', 'customer_type']:
        df_profile[col] = df_profile[col].fillna(0).astype(int)

print(f'Profile shape: {df_profile.shape}')
df_profile.head()

In [ ]:
# Step 4: Build metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_profile)

for col in DISCRETE_COLUMNS:
    metadata.update_column(col, sdtype='categorical')
for col in CONTINUOUS_COLUMNS:
    metadata.update_column(col, sdtype='numerical')

print('Metadata configured')
print(metadata.columns)

In [ ]:
# Step 5: Train CTGAN (GPU will speed this up significantly)
# Using 150 epochs instead of 300 — Colab GPU makes each epoch faster
# and 150 is sufficient for convergence on 219k rows

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

synthesizer = CTGANSynthesizer(
    metadata,
    epochs=150,
    batch_size=500,
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    pac=10,
    cuda=torch.cuda.is_available(),
    verbose=True
)

synthesizer.fit(df_profile)
print('\nTraining complete!')

In [ ]:
# Step 6: Save model + metadata
synthesizer.save('ctgan_model.pkl')
print('Saved ctgan_model.pkl')

metadata_dict = {
    'trained_date': str(date.today()),
    'training_rows': int(len(df_profile)),
    'discrete_columns': DISCRETE_COLUMNS,
    'continuous_columns': CONTINUOUS_COLUMNS,
    'column_order': PROFILE_COLUMNS,
    'los_cap': LOS_CAP,
    'market_segment_encoding': SEGMENT_ENCODE,
    'customer_type_encoding': CUSTOMER_TYPE_ENCODE
}

with open('ctgan_metadata.json', 'w') as f:
    json.dump(metadata_dict, f, indent=2)
print('Saved ctgan_metadata.json')

In [ ]:
# Step 7: Verification — 200 samples for City Hotel January
condition_df = pd.DataFrame({
    'hotel_encoded': [0] * 200,
    'month_num': [1] * 200
})

try:
    samples = synthesizer.sample_remaining_columns(condition_df)
except Exception:
    print('Conditioned sampling not available, sampling + filtering...')
    raw_samples = synthesizer.sample(num_rows=2000)
    samples = raw_samples[
        (raw_samples['hotel_encoded'].round() == 0) &
        (raw_samples['month_num'].round() == 1)
    ].head(200)

print(f'Generated {len(samples)} samples')

mean_los = samples['los'].clip(lower=1).mean()
room_tier_1_share = (samples['room_tier'].round().clip(1, 4) == 1).mean()
online_ta_share = (samples['market_segment'] == 'Online TA').mean()
non_refund_rate = (samples['is_non_refund'].round().clip(0, 1) == 1).mean()
mean_total_guests = samples['total_guests'].clip(lower=1).mean()

checks = [
    ('Mean LOS', mean_los, 2.9, 2.4, 3.4),
    ('Room tier 1 share', room_tier_1_share, 0.81, 0.70, 0.90),
    ('Online TA share', online_ta_share, 0.45, 0.35, 0.55),
    ('Non-refund rate', non_refund_rate, 0.17, 0.12, 0.22),
    ('Mean total_guests', mean_total_guests, 1.7, 1.4, 2.0),
]

print(f"\n{'Metric':<25} {'Value':>8} {'Expected':>10} {'Range':>15} {'Status':>8}")
print('-' * 75)
for name, val, expected, lo, hi in checks:
    status = '✅ PASS' if lo <= val <= hi else '❌ FAIL'
    print(f'{name:<25} {val:>8.3f} {expected:>10.2f} {f"[{lo}, {hi}]":>15} {status:>8}')

In [ ]:
# Step 8: Download both files
files.download('ctgan_model.pkl')
files.download('ctgan_metadata.json')

# After download, place both files in:
#   E:\BTP\demand_model\models\ctgan_model.pkl
#   E:\BTP\demand_model\models\ctgan_metadata.json